# 词图

In [1]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt
import math

from collections import defaultdict

In [2]:
import sys
sys.path.append('..')

In [3]:
# 项目方法
from songs.songs_libs import id_process

# 数据处理

## 分词数据
名词： n, w
动词： v
形容词： v

In [4]:
def word_count_by_pos(df, pos, words_num=30, is_starts_with=True):
    if is_starts_with:
        df_sub = df[df['pos'].str.startswith(
            pos, na=False)]
        # 歌曲数
        word_cnt = df_sub.groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos'].str.startswith(
            pos, na=False)].groupby('word')['freq'].sum().reset_index()
    else:
        word_cnt = df[df['pos']==pos]['word'].groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos']==pos].groupby('word')['freq'].sum().reset_index()
    if words_num:
        word_cnt = word_cnt.sort_values(by='song_num', ascending=False)
        res = word_cnt.head(words_num).merge(word_sum, on='word', how='left')
    else:
        res = word_cnt.merge(word_sum, on='word', how='left')
    res['order'] = 100 - res.index
    # res = res.rename(columns={
    #     'count': 'songs_num',
    # })
    return res

## 数据集提取

In [5]:
def get_subset_data(df_raw, pos, words_num, is_starts_with, is_ost=False):
    # 1. 获取符合特定词性的高频词集合
    # 假设 word_count_by_pos 返回的是一个包含 'word' 列的 DataFrame
    df_raw = df_raw.copy()  # 避免修改原始数据
    df_raw = df_raw[df_raw['word'] != '晚安' ] if pos == 't' else df_raw  # 仅在处理名词时排除“晚安”
    words_set = word_count_by_pos(df_raw,
                                  pos=pos,
                                  words_num=words_num,
                                  is_starts_with=is_starts_with)
    target_words = set(words_set['word'])  # 转为 set 匹配速度更快

    # 2. 统一词性过滤逻辑
    if is_starts_with:
        mask = df_raw['pos'].str.startswith(pos, na=False)
    else:
        mask = df_raw['pos'] == pos

    # 3. 筛选、清洗并保留必要的列
    # 链式操作：过滤词性 -> 过滤高频词 -> 执行自定义清洗
    df_subset = df_raw[mask].copy()
    df_subset = df_subset[df_subset['word'].isin(target_words)]

    # 4. 统一词性标签（既然是 Subset，统一设为传入的 pos）
    df_subset['pos'] = pos
    # 统一词的id
    df_subset = id_process(df_subset, is_ost=is_ost)

    # 排序
    df_subset = df_subset.sort_values(by=['album_order'])
    return df_subset.reset_index(drop=True)

# 词图布局

## 二分图双扇形布局

In [6]:
def generate_embracing_layout_with_expansion(nodes, stretch_factor=None):
    """
    生成开口面向直线的弧形布局
    优化：Word 节点按 degree 自上向下排列
    视觉：顶部（Degree大）间隔疏，向下（Degree小）间隔越密
    """
    max_degree = max(nodes, key=lambda x: x['degree'])['degree']
    if stretch_factor is None:
    # 定义阈值和对应的值（按降序排列）
        thresholds = [
            (80, 0.76),
            (70, 0.79),
            (60, 0.82),
            (50, 0.85)
        ]
        # 找到第一个满足条件的 factor，否则返回默认值 0.85
        stretch_factor = next((val for thresh, val in thresholds if max_degree > thresh), 0.85)

    coords = {}

    # --- 1. 几何参数设定 ---
    arc_radius = 800
    arc_span = math.pi / 1.6
    horizontal_gap = 300

    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height

    # --- 2. Word 节点 (顶部疏，底部密) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'],
                        key=lambda x: x['degree'],
                        reverse=True)
    num_words = len(word_nodes)

    target_word_height = arc_total_height * 0.9
    
    # stretch_factor 说明：
    # 指数越小（如 0.4-0.6），顶部节点被推开的幅度越大，底部压缩越厉害。
    # 指数 = 1.0 时，为等间距分布。

    for i, node in enumerate(word_nodes):
        # 归一化进度 t: 从 0 (顶部) 到 1 (底部)
        t = i / (num_words - 1) if num_words > 1 else 0
        
        # 核心逻辑：利用幂函数特性映射 Y 坐标
        # 我们希望在 t 较小时 y 变化快，t 较大时 y 变化慢
        # 公式：y = (t^p) * 总高度 - half_height
        # 这样当 t=0 时 y=-half_height; 当 t=1 时 y=half_height
        y_val = (t ** stretch_factor) * target_word_height - (target_word_height / 2)

        coords[node['id']] = (0, round(y_val, 2))

    # --- 3. Song 节点 ---
    song_stretch_factor = 0.9  
    song_nodes = [n for n in nodes if n['type'] != 'word']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]

    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1},  # 右侧弧
        {"c_x": 0, "dir": -1}  # 左侧弧
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)

        for row_idx, node in enumerate(col_items):
            # 1. 归一化进度 t: 从 0 (最上方) 到 1 (最下方)
            t = row_idx / (num_in_col - 1) if num_in_col > 1 else 0.5
            
            # 2. 映射逻辑：将 0~1 映射到 -1~1，进行幂运算后再映射回角度
            # 为了实现“中间紧凑”，我们需要一个在 0.5 附近导数较小的函数
            # 公式：t_shifted = (2t - 1) -> 范围变为 [-1, 1]
            # t_new = sign(t_s) * |t_s| ^ (1 / factor)  注意：1/0.9 > 1，会让中间更平缓
            t_shifted = 2 * t - 1
            # 使用 math.copysign 处理正负号，应用幂次让靠近 0 的值更密集
            # 指数 > 1 会拉开两端，压缩中间。 1/0.9 ≈ 1.11
            t_warped = math.copysign(abs(t_shifted) ** (1 / song_stretch_factor), t_shifted)
            
            # 3. 将 t_warped 从 [-1, 1] 映射回弧度范围 [arc_span/2, -arc_span/2]
            angle = -(t_warped * (arc_span / 2))
            
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))

    return coords

## 词-歌曲图布局

In [7]:
def get_word_song_subset_graph(G_words_subset, df_subset, is_ost=False, stretch_factor=None):
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'song',
        }
        nodes_subset.append(n_dict)
    pos =  generate_embracing_layout_with_expansion(nodes_subset, stretch_factor)
    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_subset[df_subset['word_id'] ==
                                            node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] = {'cluster': '词'}
        elif 'song' in node:
            nodes_dict['label'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['song_name_unique'].values[0]
            nodes_dict['node_type'] = 'song'
            nodes_dict['album_id'] = df_subset[
                df_subset['song_id_unique'] ==
                node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['album_order'] = int(df_subset[
                df_subset['song_id_unique'] == node]['album_order'].values[0])
            nodes_dict['tv_name'] = ""
            if is_ost == True:
                nodes_dict['tv_name'] = df_subset[
                    df_subset['song_id_unique'] ==
                    node]['tv_name'].values[0]
            nodes_dict['data'] = {
                'cluster':
                df_subset[df_subset['song_id_unique'] == node]
                ['album_fixed'].values[0]
            }
        word_graph_dict['nodes'].append(nodes_dict)
    for edge in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edge[0]
        edges_dict['target'] = edge[1]
        for i in edge:
            if 'song' in i:
                edges_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                                i]['album_fixed'].values[0]
                edges_dict['album_order'] = int(df_subset[
                    df_subset['song_id_unique'] == i]['album_order'].values[0])
            # else:
            #     edges_dict['album'] = ""
            #     edges_dict['album_order'] = ""

        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

## 词-专辑图布局
默认力引导图

In [8]:
def get_word_album_subset_graph(G_words_subset,
                                df_subset,
                                stretch_factor=None):
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'album',
        }
        nodes_subset.append(n_dict)
    word_song_counts = df_subset.groupby('word_id').agg({
        'song_id': 'count',  # 计算每个词对应的不同song_id数量
    }).reset_index().rename(columns={
        'song_id': 'song_id_count',
    })
    # pos =  generate_embracing_layout_with_expansion(nodes_subset, stretch_factor)
    pos = nx.spring_layout(G_words_subset, iterations=100, seed=42)  # 使用 spring_layout 生成节点位置
    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_subset[df_subset['word_id'] ==
                                            node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] = {'cluster': '词'}
            nodes_dict['song_id_count'] = int(word_song_counts[word_song_counts['word_id'] == node]['song_id_count'].values[0])
        elif 'album' in node:
            nodes_dict['size'] = words_degree[node]
            nodes_dict['label'] = df_subset[df_subset['album_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['node_type'] = 'album'
            nodes_dict['album_id'] = df_subset[
                df_subset['album_id_unique'] ==
                node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_subset[df_subset['album_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['album_order'] = int(df_subset[
                df_subset['album_id_unique'] == node]['album_order'].values[0])
            # nodes_dict['tv_name'] = ""
            # if is_ost == True:
            #     nodes_dict['tv_name'] = df_subset[
            #         df_subset['album_id_unique'] ==
            #         node]['tv_name'].values[0]
            nodes_dict['data'] = {
                'cluster':
                df_subset[df_subset['album_id_unique'] == node]
                ['album_fixed'].values[0]
            }
        word_graph_dict['nodes'].append(nodes_dict)
    for edge in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edge[0]
        edges_dict['target'] = edge[1]
        edges_dict['weight'] = G_words_subset[edge[0]][edge[1]]['weight']
        for i in edge:
            if 'album' in i:
                edges_dict['album'] = df_subset[df_subset['album_id_unique'] ==
                                                i]['album_fixed'].values[0]
                edges_dict['album_order'] = int(
                    df_subset[df_subset['album_id_unique'] ==
                              i]['album_order'].values[0])
            # else:
            #     edges_dict['album'] = ""
            #     edges_dict['album_order'] = ""

        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

# 计算与json输出

## 词-曲

In [9]:
def calculate_words_songs_bip_graph(path_prefix, pos_type="n", words_num=50, is_starts_with=True, is_ost=False, stretch_factor=None):
    df_words_raw = pd.read_csv(path_prefix + "cleared_words_data.csv")
    df_subset = get_subset_data(df_words_raw, pos=pos_type, words_num=words_num, is_starts_with=is_starts_with, is_ost=is_ost)
    G_words_subset = nx.from_pandas_edgelist(df_subset, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
    # 添加数据信息
    pos_dict = {'n': '名词', 'a': '形容词', 'v': '动词', 't': '时间词'}
    album_type = '录音室&精选辑' if path_prefix == "data/mayday/" else '录音室专辑'
    album_type = '影视剧OST歌曲' if path_prefix == "data/liuyuning/" else album_type
    singer = df_subset['artist_name'].values[0]
    data_info = {
        'singer': singer,
        'title': f'{album_type}',
        'pos': pos_dict[pos_type],
        'all_num': df_words_raw['song_id'].nunique(),
        'has_pos_num': df_subset['song_id_unique'].nunique(),
        'word_num': words_num,
        'num_unit': '首',
        'nodes_type': f'{pos_dict[pos_type]}-歌曲',
    }
    word_graph_dict = get_word_song_subset_graph(G_words_subset, df_subset, stretch_factor)
    word_graph_dict['data_info'] = data_info
    with open(path_prefix+f'{pos_type}_word_song_graph_data.json', 'w', encoding='utf-8') as f:
        json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)

## 词-专辑

In [10]:
def calculate_words_albums_bip_graph(path_prefix, pos_type="n", words_num=50, is_starts_with=True, is_ost=False, stretch_factor=None):
    df_words_raw = pd.read_csv(path_prefix + "cleared_words_data.csv")
    df_subset = get_subset_data(df_words_raw, pos=pos_type, words_num=words_num, is_starts_with=is_starts_with, is_ost=is_ost)
    # 1. 使用 groupby 统计每对节点出现的次数，并重命名为 'weight'
    df_weighted = df_subset.groupby(['word_id', 'album_id_unique']).size().reset_index(name='weight')

    # 2. 正常创建图，此时 edge_attr 会自动读取 'weight' 列
    G_words_album_subset = nx.from_pandas_edgelist(
        df_weighted, 
        'word_id', 
        'album_id_unique', 
        edge_attr='weight', 
        create_using=nx.Graph()
    )
    # 添加数据信息
    pos_dict = {'n': '名词', 'a': '形容词', 'v': '动词', 't': '时间词'}
    album_type = '录音室&精选辑' if path_prefix == "data/mayday/" else '录音室专辑'
    album_type = '影视剧OST歌曲' if path_prefix == "data/liuyuning/" else album_type
    singer = df_subset['artist_name'].values[0]
    data_info = {
        'singer': singer,
        'title': f'{album_type}',
        'pos': pos_dict[pos_type],
        'word_num': words_num,
        'all_num': df_words_raw['album_id'].nunique(),
        'has_pos_num': df_subset['album_id_unique'].nunique(),
        'num_unit': '专辑',
        'nodes_type': f'{pos_dict[pos_type]}-专辑',
    }
    word_graph_dict = get_word_album_subset_graph(G_words_album_subset, df_subset, stretch_factor=0.8)
    word_graph_dict['data_info'] = data_info
    with open(path_prefix+f'{pos_type}_word_album_graph_data.json', 'w', encoding='utf-8') as f:
        json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)

# main

In [11]:
# file_path_prefix = "data/jaychou/"
file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"

## 词-曲关系网

In [12]:
for pos in ['n', 'a', 'v', 't']:
    calculate_words_songs_bip_graph(file_path_prefix, pos_type=pos, words_num=20, is_starts_with=True, is_ost=False, stretch_factor=0.8)

## 词-专辑关系网

In [13]:
for pos in ['n', 'a', 'v', 't']:
    calculate_words_albums_bip_graph(file_path_prefix, pos_type=pos, words_num=20, is_starts_with=True, is_ost=False, stretch_factor=None)

In [14]:
645/3*4

860.0